# Predicting Protein Prediction

Consider as starting point for this exercise a UCI Protein Structure dataset (CASP). The goal is to predict the RMSD of protein structure predictions from nine abstract physicochemical descriptors.

#### Tasks:
1. Inspect the data.
2. Build feature matrix and target vector.
3. Choose one regression model and optimise it.
4. Report approximate training and test time.
5. Evaluate with **RMSE**.
6. Respond to the discussion points.

In [3]:
import time

import numpy as np
import pandas as pd

from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import train_test_split

Load and investigate the data

In [28]:
df = pd.read_csv("CASP.csv")

display(df)
display(df.isna().sum())
df.describe().round(2)

,RMSD,F1,F2,F3,F4,F5,F6,F7,F8,F9
0,17.284,13558.30,4305.35,0.31754,162.1730,1.872791e+06,215.3590,4287.87,102,27.0302
1,6.021,6191.96,1623.16,0.26213,53.3894,8.034467e+05,87.2024,3328.91,39,38.5468
2,9.275,7725.98,1726.28,0.22343,67.2887,1.075648e+06,81.7913,2981.04,29,38.8119
3,15.851,8424.58,2368.25,0.28111,67.8325,1.210472e+06,109.4390,3248.22,70,39.0651
4,7.962,7460.84,1736.94,0.23280,52.4123,1.021020e+06,94.5234,2814.42,41,39.9147
...,...,...,...,...,...,...,...,...,...,...
45725,3.762,8037.12,2777.68,0.34560,64.3390,1.105797e+06,112.7460,3384.21,84,36.8036
45726,6.521,7978.76,2508.57,0.31440,75.8654,1.116725e+06,102.2770,3974.52,54,36.0470
45727,10.356,7726.65,2489.58,0.32220,70.9903,1.076560e+06,103.6780,3290.46,46,37.4718
45728,9.791,8878.93,3055.78,0.34416,94.0314,1.242266e+06,115.1950,3421.79,41,35.6045


RMSD    0
F1      0
F2      0
F3      0
F4      0
F5      0
F6      0
F7      0
F8      0
F9      0
dtype: int64

,RMSD,F1,F2,F3,F4,F5,F6,F7,F8,F9
count,45730.00,45730.00,45730.00,45730.00,45730.00,45730.00,45730.00,45730.00,45730.00,45730.00
mean,7.75,9871.60,3017.37,0.30,103.49,1368299.02,145.64,3989.76,69.98,34.52
std,6.12,4058.14,1464.32,0.06,55.42,564036.69,70.00,1993.57,56.49,5.98
min,0.00,2392.05,403.50,0.09,10.31,319490.22,31.97,0.00,0.00,15.23
25%,2.31,6936.68,1979.04,0.26,63.56,953591.22,94.76,3165.32,31.00,30.42
50%,5.03,8898.80,2668.16,0.30,87.74,1237219.06,126.18,3840.17,54.00,35.30
75%,13.38,12126.15,3786.41,0.34,133.65,1690919.95,181.47,4644.19,91.00,38.87
max,21.00,40034.90,15312.00,0.58,369.32,5472011.41,598.41,105948.17,350.00,55.30


Build feature matrix and target vector. Add scaling if needed for your model.

In [18]:
X = df.drop(columns=["RMSD"])
y = df["RMSD"]

X_train_full, X_test, y_train_full, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
)

X_train, X_val, y_train, y_val = train_test_split(
    X_train_full,
    y_train_full,
    test_size=0.2,
    random_state=42,
)

# No scaling is used here because GradientBoostingRegressor is tree-based
# and therefore not sensitive to feature scale in the same way as linear models or SVR.
print(y_train.shape, y_val.shape, y_train.shape)
print(X_train.shape, X_val.shape, X_test.shape)

(29267,) (7317,) (29267,)
(29267, 9) (7317, 9) (9146, 9)


Choose a Regression model, build, train and optimise

In [10]:
candidate_params = [
    {"n_estimators": 200, "learning_rate": 0.05, "max_depth": 3, "min_samples_leaf": 3, "subsample": 0.9},
    {"n_estimators": 300, "learning_rate": 0.05, "max_depth": 3, "min_samples_leaf": 3, "subsample": 0.9},
    {"n_estimators": 250, "learning_rate": 0.05, "max_depth": 4, "min_samples_leaf": 3, "subsample": 0.9},
    {"n_estimators": 300, "learning_rate": 0.05, "max_depth": 4, "min_samples_leaf": 3, "subsample": 0.9},
]

search_results = []

for params in candidate_params:
    model = GradientBoostingRegressor(random_state=42, **params)
    t0 = time.perf_counter()
    model.fit(X_train, y_train)
    fit_time = time.perf_counter() - t0
    val_pred = model.predict(X_val)

    search_results.append(
        {
            **params,
            "val_rmse": np.sqrt(mean_squared_error(y_val, val_pred)),
            "fit_time_s": fit_time,
        }
    )

results_df = pd.DataFrame(search_results).sort_values("val_rmse")
results_df

,n_estimators,learning_rate,max_depth,min_samples_leaf,subsample,val_rmse,fit_time_s
3,300,0.05,4,3,0.9,4.304808,32.824025
2,250,0.05,4,3,0.9,4.358981,26.610144
1,300,0.05,3,3,0.9,4.535469,25.279456
0,200,0.05,3,3,0.9,4.671447,17.083812


Evaluate your best model (RMSE). Take note of training and test time (approximate).

In [15]:
best_params = results_df.iloc[0][["n_estimators", "learning_rate", "max_depth", "min_samples_leaf", "subsample"]].to_dict()
best_params["n_estimators"] = int(best_params["n_estimators"])
best_params["max_depth"] = int(best_params["max_depth"])
best_params["min_samples_leaf"] = int(best_params["min_samples_leaf"])

best_model = GradientBoostingRegressor(random_state=42, **best_params)

print(best_params)

t0 = time.perf_counter()
best_model.fit(X_train_full, y_train_full)
final_fit_time = time.perf_counter() - t0

t1 = time.perf_counter()
y_train_pred = best_model.predict(X_train_full)
y_test_pred = best_model.predict(X_test)
final_predict_time = time.perf_counter() - t1

train_rmse = np.sqrt(mean_squared_error(y_train_full, y_train_pred))
test_rmse = np.sqrt(mean_squared_error(y_test, y_test_pred))
train_r2 = r2_score(y_train_full, y_train_pred)
test_r2 = r2_score(y_test, y_test_pred)

final_results = pd.DataFrame(
    [
        {
            "train_rmse": train_rmse,
            "test_rmse": test_rmse,
            "train_r2": train_r2,
            "test_r2": test_r2,
            "fit_time_s": final_fit_time,
            "predict_time_s": final_predict_time,
        }
    ]
)

final_results.round(4)

{'n_estimators': 300, 'learning_rate': 0.05, 'max_depth': 4, 'min_samples_leaf': 3, 'subsample': 0.9}


,train_rmse,test_rmse,train_r2,test_r2,fit_time_s,predict_time_s
0,4.1537,4.3573,0.5385,0.4955,41.0957,0.1796


#### Discussion points
1) Discuss your choice of model class.
2) How did you optimise your model? How did the best model perform?
3) How much time was needed for training the model and evaluations (approximation is enough)?
4) What limitations or shortcomings did you identify? What would be ideas to remedy or circumvent them?
5) In all its abstraction, what do the predictions of your model tell you?

#### Short Answers

1. I chose `GradientBoostingRegressor` because the CASP descriptors are likely related to RMSD in a non-linear way, and boosting can model feature interactions better than a simple linear regressor. It also works well on medium-to-large tabular data and does not require scaling.
2. I optimised the model with a small validation sweep over `n_estimators`, `max_depth`, and related regularisation settings such as `subsample` and `min_samples_leaf`. The best validation setting was `n_estimators=300`, `learning_rate=0.05`, `max_depth=4`, `min_samples_leaf=3`, `subsample=0.9`. Retrained on the full training split, it achieved a train RMSE of about `4.1537` and a test RMSE of about `4.3573` with test R2 around `0.4955`.
3. The tuning fits took roughly `17-34` seconds per candidate on this machine. The final model fit on the full training split took about `41.10` seconds, and predicting both train and test targets took about `0.17` seconds.
4. The model still shows some train-test gap, so there is remaining overfitting and limited predictive power. Another limitation is that the features are abstract engineered descriptors, so interpretability is limited. Possible improvements would be broader hyperparameter search, cross-validation, more expressive ensemble methods, or neural-network / graph-based models if runtime permits.
5. The predictions estimate how far a predicted protein structure is expected to deviate from the true structure. In practical terms, the model gives a data-driven estimate of structure quality: lower predicted RMSD suggests a more accurate structure prediction, while higher RMSD suggests less reliable predictions.